# SGPD (GSE145361) cohort

In [1]:
import os
import gc
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, GlobalAttention
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

/opt/conda/envs/rapids-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
FUNCTIONAL_MAP = {
    'TSS200': 0, 'TSS1500': 1, '1stExon': 2, 
    "5'UTR": 3, 'Body': 4, "3'UTR": 5, 'Other': 6
}

## Build the Chromosome topologies

In [3]:
def build_chromosome_topologies(manifest_df, common_probes, max_linear_dist_bp=1000):
    manifest = manifest_df[manifest_df['IlmnID'].isin(common_probes)].copy()
    manifest['CHR'] = manifest['CHR'].astype(str).str.replace('chr', '')
    manifest = manifest[manifest['CHR'].isin([str(i) for i in range(1, 23)])]
    manifest['MAPINFO'] = manifest['MAPINFO'].astype(int)
    
    chr_topologies = {}
    
    for c in range(1, 23):
        chr_str = str(c)
        sub = manifest[manifest['CHR'] == chr_str].sort_values(by='MAPINFO').reset_index(drop=True)
        probe_list = sub['IlmnID'].tolist()
        probe_to_idx = {p: i for i, p in enumerate(probe_list)}
        positions = sub['MAPINFO'].values
        n_nodes = len(probe_list)
        
        # A. Encode primary functional annotation
        func_labels = np.full(n_nodes, FUNCTIONAL_MAP['Other'], dtype=np.int64)
        for idx, row in sub.iterrows():
            grps = str(row.get('UCSC_RefGene_Group', '')).split(';')
            if grps and grps[0] in FUNCTIONAL_MAP:
                func_labels[idx] = FUNCTIONAL_MAP[grps[0]]
                
        # B. EXPLICIT SELF LOOPS (Fixes Empty Edge Crash & Mathematically sound for GAT)
        edges = [[i, i] for i in range(n_nodes)]
        
        # C. 1D Linear Adjacency Edges
        for i in range(n_nodes - 1):
            if 0 < (positions[i+1] - positions[i]) <= max_linear_dist_bp:
                edges.append([i, i + 1])
                edges.append([i + 1, i])
                
        # D. Shared Genic Annotation Edges
        gene_groups = {}
        for idx, row in sub.iterrows():
            genes = str(row.get('UCSC_RefGene_Name', '')).split(';')
            if genes and genes[0] not in ('', 'nan'):
                gene = genes[0]
                gene_groups.setdefault(gene, []).append(idx)
                
        for members in gene_groups.values():
            if 1 < len(members) <= 50:
                for i in range(len(members)):
                    for j in range(i + 1, len(members)):
                        edges.append([members[i], members[j]])
                        edges.append([members[j], members[i]])
                        
        # Because we initialized with self-loops, 'edges' is guaranteed non-empty
        edge_arr = np.unique(np.array(edges), axis=0).T
        edge_index = torch.tensor(edge_arr, dtype=torch.long)
            
        chr_topologies[c] = {
            'probes': probe_list,
            'edge_index': edge_index,
            'func_type': torch.tensor(func_labels, dtype=torch.long),
            'n_nodes': n_nodes
        }
        
    return chr_topologies

## PyTorch dataset and Trasposed Collator

In [4]:
LABEL_MAP = {'Control': 0.0, 'PD': 1.0}

class WholeBloodMethylationDataset(Dataset):
    def __init__(self, m_matrix_df, pheno_df, cell_cols, chr_topologies):
        self.sample_ids = pheno_df.index.tolist()
        
        # Explicit label encoding for BCEWithLogitsLoss
        labels = pheno_df['Sample_Group'].map(LABEL_MAP)
        if labels.isna().any():
            unknown_labels = sorted(pheno_df.loc[labels.isna(), 'Sample_Group'].astype(str).unique().tolist())
            raise ValueError(f"Unexpected Sample_Group values: {unknown_labels}")
        self.labels = labels.to_numpy(dtype=np.float32)
        
        self.cell_props = pheno_df[cell_cols].values.astype(np.float32)
        self.chr_topologies = chr_topologies
        
        self.chr_m_values = {}
        for c in range(1, 23):
            probes = chr_topologies[c]['probes']
            # Safe slice: guarantees no missing probes because df was pre-subsetted
            self.chr_m_values[c] = m_matrix_df.loc[self.sample_ids, probes].values.astype(np.float32)
            
    def __len__(self):
        return len(self.sample_ids)
        
    def __getitem__(self, idx):
        chr_graphs = []
        for c in range(1, 23):
            raw_x = torch.from_numpy(self.chr_m_values[c][idx]).unsqueeze(1)
            topo = self.chr_topologies[c]
            
            data = Data(
                x=raw_x,
                edge_index=topo['edge_index'],
                func_type=topo['func_type']
            )
            chr_graphs.append(data)
            
        u = torch.from_numpy(self.cell_props[idx])
        y = torch.tensor(self.labels[idx], dtype=torch.float32)
        return chr_graphs, u, y


def chromosome_collate_fn(batch):
    batched_chromosomes = []
    graphs_per_sample = [item[0] for item in batch]
    u_tensor = torch.stack([item[1] for item in batch])
    y_tensor = torch.stack([item[2] for item in batch])
    
    for chr_idx in range(22):
        chr_list = [graphs[chr_idx] for graphs in graphs_per_sample]
        batched_chromosomes.append(Batch.from_data_list(chr_list))
        
    return batched_chromosomes, u_tensor, y_tensor

## Chromosome-Parallel GAT Architecture

In [5]:
class ChromosomeParallelGAT(nn.Module):
    def __init__(self, num_node_classes=7, chr_embed_dim=16, cell_prop_dim=6):
        super(ChromosomeParallelGAT, self).__init__()
        
        self.func_embedding = nn.Embedding(num_embeddings=num_node_classes, embedding_dim=8)
        
        # add_self_loops=False because we explicitly defined them in the topology builder
        self.gat1 = GATConv(in_channels=9, out_channels=8, heads=2, concat=True, add_self_loops=False)
        self.gat2 = GATConv(in_channels=16, out_channels=chr_embed_dim, heads=1, concat=True, add_self_loops=False)
        
        self.gate_nn = nn.Sequential(
            nn.Linear(chr_embed_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
        self.pool = GlobalAttention(gate_nn=self.gate_nn)
        
        fused_dim = (22 * chr_embed_dim) + cell_prop_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
        
    def forward(self, batched_chrs, u_cells):
        chr_embeddings = []
        for c_idx in range(22):
            batch = batched_chrs[c_idx]
            
            emb_func = self.func_embedding(batch.func_type)
            node_feat = torch.cat([batch.x, emb_func], dim=1)
            
            h = torch.relu(self.gat1(node_feat, batch.edge_index))
            h = torch.relu(self.gat2(h, batch.edge_index))
            
            chr_emb = self.pool(h, batch.batch)
            chr_embeddings.append(chr_emb)
            
        genome_vector = torch.cat(chr_embeddings, dim=1)
        fused = torch.cat([genome_vector, u_cells], dim=1)
        return self.classifier(fused)

## Data preps
___
Here we need to create distinct sets of pheno-data and CpG m-values. Those will be split off from the parquet file.

In [6]:
m_matrix_full = pd.read_parquet("/workspace/data/sgpd/GSE145361_data_corrected.parquet")
m_values_df = m_matrix_full.set_index("Sample_Name")

In [7]:
all_cols = m_values_df.columns.to_list()
probe_cols = [c for c in all_cols if c.startswith('cg') or (c.startswith('ch') and not c.startswith('cha'))]

In [8]:
probe_set = set(probe_cols)
pheno_cols = [c for c in all_cols if c not in probe_set]

In [ ]:
pheno_df = m_values_df[pheno_cols].copy()

cat_cols = pheno_df.select_dtypes(include=["category"]).columns
for c in cat_cols:
    pheno_df[c] = pheno_df[c].astype("str")

pheno_out = "/workspace/data/sgpd/GSE145361_pheno_data.parquet"
pheno_df.to_parquet(pheno_out)
print(f"Wrote pheno_df to {pheno_out}")

Wrote pheno_df to /workspace/data/sgpd/GSE111629_pheno_data.parquet


In [10]:
manifest_path = "/workspace/data/infinium450k_manifest.parquet"
manifest_df = pd.read_parquet(manifest_path)

## 5-Fold CV

In [ ]:

LABEL_MAP = {'Control': 0, 'PD': 1}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing GAT Pipeline on: {device}")

if pheno_df.index.name != "Sample_Name":
    pheno_df = pheno_df.set_index("Sample_Name")
    
cell_cols = ['CD8T', 'CD4T', 'NK', 'Bcell', 'Mono', 'Gran']
# Index is already set 
# m_values_df.set_index("Sample_Name", inplace=True)

# Strict validation to prevent index mismatch
common_probes = list(set(m_values_df.columns).intersection(set(manifest_df['IlmnID'])))
m_values_df = m_values_df[common_probes] # Discard unused columns to save RAM early
print(f"Total overlapping autosome probes: {len(common_probes):,}")

print("Building chromosome 1D adjacency and genic graphs...")
chr_topologies = build_chromosome_topologies(manifest_df, common_probes)

y = pheno_df['Sample_Group'].map(LABEL_MAP)
if y.isna().any():
    unknown_labels = sorted(pheno_df.loc[y.isna(), 'Sample_Group'].astype(str).unique().tolist())
    raise ValueError(f"Unexpected Sample_Group values: {unknown_labels}")
y = y.to_numpy(dtype=np.int64)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_aucs, pr_aucs = [], []
batch_size = 16  
epochs = 40
lr = 2e-4

for fold, (train_idx, test_idx) in enumerate(skf.split(pheno_df, y)):
    print(f"\n--- Fold {fold + 1} ---")
    train_pheno = pheno_df.iloc[train_idx]
    test_pheno = pheno_df.iloc[test_idx]
    
    train_ds = WholeBloodMethylationDataset(m_values_df, train_pheno, cell_cols, chr_topologies)
    test_ds = WholeBloodMethylationDataset(m_values_df, test_pheno, cell_cols, chr_topologies)
    
    train_loader = DataLoader(train_ds, 
                              batch_size=batch_size, 
                              shuffle=True, 
                              collate_fn=chromosome_collate_fn,
                              num_workers=4,
                              pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=chromosome_collate_fn)
    
    model = ChromosomeParallelGAT().to(device)
    
    # Explicit dtype declaration for pos_weight
    train_labels = train_pheno['Sample_Group'].map(LABEL_MAP)
    if train_labels.isna().any():
        unknown_labels = sorted(train_pheno.loc[train_labels.isna(), 'Sample_Group'].astype(str).unique().tolist())
        raise ValueError(f"Unexpected Sample_Group values in training fold: {unknown_labels}")
    train_labels = train_labels.to_numpy(dtype=np.float32)
    positives = float((train_labels == 1.0).sum())
    negatives = float((train_labels == 0.0).sum())
    if positives == 0:
        raise ValueError("Training fold has no positive samples, cannot compute pos_weight")
    pos_weight = torch.tensor([negatives / positives], device=device, dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    
    # Training
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for batched_chrs, u_cells, batch_y in train_loader:
            
            # Asynchronous memory transfer
            batched_chrs = [c.to(device, non_blocking=True) for c in batched_chrs]
            u_cells = u_cells.to(device, non_blocking=True)
            batch_y = batch_y.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            logits = model(batched_chrs, u_cells).squeeze(1)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
            # VRAM Fragmentation Protection
            del batched_chrs, u_cells, batch_y, logits, loss
            
    # Evaluation
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batched_chrs, u_cells, batch_y in test_loader:
            batched_chrs = [c.to(device, non_blocking=True) for c in batched_chrs]
            u_cells = u_cells.to(device, non_blocking=True)
            
            logits = model(batched_chrs, u_cells).squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            preds.extend(probs)
            truths.extend(batch_y.numpy())
            
            del batched_chrs, u_cells, logits
            
    fold_roc = roc_auc_score(truths, preds)
    fold_pr = average_precision_score(truths, preds)
    roc_aucs.append(fold_roc)
    pr_aucs.append(fold_pr)
    print(f"Fold {fold + 1} | ROC AUC: {fold_roc:.4f} | PR AUC: {fold_pr:.4f}")

    # Save the model weights for downstream biological extraction
    torch.save(model.state_dict(), f"/workspace/results/sgpd/gat_fold_{fold + 1}.pt")

    del model, optimizer, train_ds, test_ds
    torch.cuda.empty_cache()
    gc.collect()
    
print("\n" + "=" * 40)
print(f"Mean GAT ROC AUC: {np.mean(roc_aucs):.4f} ± {np.std(roc_aucs):.4f}")
print(f"Mean GAT PR AUC:  {np.mean(pr_aucs):.4f} ± {np.std(pr_aucs):.4f}")

Executing GAT Pipeline on: cuda
Total overlapping autosome probes: 421,350
Building chromosome 1D adjacency and genic graphs...

--- Fold 1 ---


/tmp/ipykernel_1583/2041644710.py:16: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn=self.gate_nn)


: 